# Modelado de PM2.5 — entrega final

## 1. Objetivo

Este notebook construye la tabla de predictores y evalúa los modelos de la entrega final. Responde las dos preguntas de investigación:

1. ¿Puede un modelo de Gradient Boosting, con ventanas de PM2.5 de los días previos y variables de calendario, predecir con un día de anticipación la media diaria de PM2.5 por estación con menor error que la persistencia y anticipar mejor los días en Nivel de Prevención?
2. ¿Cuánto mejora ese pronóstico al añadir la meteorología observada de SIATA, y qué variables explican sus predicciones según SHAP?

Esta primera versión cubre la tabla de predictores y el desempeño de la persistencia, la línea base que todo modelo debe superar. Requiere haber ejecutado los pasos de puesta en marcha del README (descargas y `scripts/build_meteo_hourly.py`).

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data import load_pm25_files
from src.data.stations import PM25_TO_METEO
from src.features import CALENDAR_FEATURES, METEO_FEATURES, PM25_FEATURES, TARGET, build_features

DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'pm25'
METEO_HOURLY = PROJECT_ROOT / 'data' / 'interim' / 'meteo_hourly.csv'
PREVENTION_THRESHOLD = 38.0  # Nivel de Prevención PM2.5 24 h, Resolución 2254 de 2017, Tabla 4
TEST_YEARS = [2022, 2023, 2024, 2025]

## 2. Carga

Se usan las mismas 16 estaciones y la misma ventana que en el EDA (2018–2025). La carga empieza el 25 de diciembre de 2017 solo para que los rezagos de los primeros días de 2018 tengan datos; esos días previos se descartan después. La meteorología es la tabla horaria construida por `scripts/build_meteo_hourly.py`, que ya descarta los minutos marcados como dudosos y exige 45 minutos válidos por hora.

In [2]:
LOAD_START, WINDOW_START, WINDOW_END = '2017-12-25', '2018-01-01', '2025-12-31'

pm25_wide = load_pm25_files(DATA_DIR)
pm25_wide = pm25_wide[pm25_wide['fecha_hora'].between(LOAD_START, f'{WINDOW_END} 23:00')]
pm25_hourly = pm25_wide.melt(
    id_vars='fecha_hora', value_vars=list(PM25_TO_METEO), var_name='station_id', value_name='pm25'
)
meteo_hourly = pd.read_csv(METEO_HOURLY, parse_dates=['fecha_hora'])

print('PM2.5 horario (estación-hora):', pm25_hourly.shape)
print('Meteorología horaria:', meteo_hourly.shape)
print('Horas con presión de 0 hPa:', int((meteo_hourly['pr'] == 0).sum()))

PM2.5 horario (estación-hora): (1124736, 3)
Meteorología horaria: (801839, 16)
Horas con presión de 0 hPa: 7


La bandera de calidad de SIATA no marca las 7 horas con presión de 0 hPa detectadas en el EDA. Una presión de 0 hPa no es física (en el valle ronda 840 hPa), así que `build_features` las trata como faltantes antes de calcular la media diaria. Era un pendiente abierto desde la Entrega 1.

## 3. Tabla de predictores

Cada fila es una **estación en un día `d`**. Todo predictor usa información observada hasta el final del día `d`; el target es la media del día `d + 1`. La construcción está en `src/features/build_features.py` y sus pruebas en `tests/test_build_features.py`, que incluyen una que verifica que cambiar cualquier dato del día `d + 1` solo cambia el target.

| Grupo | Predictores | Qué representan |
| --- | --- | --- |
| PM2.5 de hoy | `pm25_mean`, `pm25_max`, `pm25_evening_mean` | Media del día (≥18 h válidas), máximo horario y media de 18:00 a 23:00 (≥4 h), la parte del día más cercana a mañana. |
| Ventanas de tiempo | `pm25_lag1` … `pm25_lag6` | Media diaria de 1 a 6 días antes. Con `pm25_mean` cubren la última semana. |
| Ventanas móviles | `pm25_roll3_mean`, `pm25_roll7_mean`, `pm25_roll7_std`, `pm25_diff1` | Nivel y variabilidad de los últimos 3 y 7 días, y cambio respecto a ayer. |
| Meteorología del día | `t`, `h`, `pr`, `vv`, `vv_max`, `p`, `p_sum3` | Temperatura, humedad, presión y viento medios; ráfaga máxima; lluvia del día y de los últimos 3 días (entrada que usó Baena-Salazar et al.). |
| Calendario de mañana | `target_month`, `target_day_of_week` | Mes y día de la semana del día pronosticado, que se conocen de antemano. |

Las ventanas móviles admiten algunos días faltantes (2 de 3 y 5 de 7); los rezagos y la lluvia de 3 días no. No se imputa nada.

In [3]:
features = build_features(pm25_hourly, meteo_hourly, PM25_TO_METEO, LOAD_START, WINDOW_END)
features = features[features['date'] >= WINDOW_START].reset_index(drop=True)
supervised = features.dropna(subset=['pm25_mean', TARGET]).copy()
supervised['year'] = supervised['date'].dt.year

print('Tabla estación-día:', features.shape)
print('Con media de hoy:', int(features['pm25_mean'].notna().sum()))
print('Tabla supervisada (media de hoy y target):', len(supervised))
display(supervised.head())

Tabla estación-día: (46752, 27)
Con media de hoy: 45152
Tabla supervisada (media de hoy y target): 44399


,station_id,date,codigo,valid_hours,pm25_mean,pm25_lag1,pm25_lag2,pm25_lag3,pm25_lag4,pm25_lag5,...,h,pr,vv,vv_max,p,p_sum3,target_month,target_day_of_week,pm25_mean_d_plus_1,year
0,BAR-TORR,2018-01-01,82,23,17.831593,10.133717,11.499272,9.370924,12.287847,11.286822,...,74.700417,865.101806,1.610556,8.9,10.97,NaN,1,1,13.014660,2018
1,BAR-TORR,2018-01-02,82,23,13.014660,17.831593,10.133717,11.499272,9.370924,12.287847,...,81.694641,864.424681,1.273681,10.6,46.63,NaN,1,2,12.264085,2018
2,BAR-TORR,2018-01-03,82,23,12.264085,13.014660,17.831593,10.133717,11.499272,9.370924,...,77.293607,863.840467,1.534718,17.7,47.25,104.85,1,3,10.933441,2018
3,BAR-TORR,2018-01-04,82,24,10.933441,12.264085,13.014660,17.831593,10.133717,11.499272,...,74.222917,865.056111,2.053194,8.0,1.83,95.71,1,4,16.933715,2018
4,BAR-TORR,2018-01-05,82,24,16.933715,10.933441,12.264085,13.014660,17.831593,10.133717,...,80.900625,865.985972,1.326528,6.7,11.98,61.06,1,5,13.421844,2018


Las cifras coinciden con el EDA: 46.752 combinaciones estación-día, 45.152 con media diaria y 44.399 con media de hoy y target de mañana. La tabla de predictores no pierde ni añade registros respecto a la caracterización de la Entrega 1.

## 4. Faltantes por predictor

Gradient Boosting de scikit-learn en su versión `HistGradientBoostingRegressor` acepta valores faltantes: en cada división aprende hacia qué rama enviarlos. Por eso se conservan los registros con algún predictor vacío en vez de descartarlos o imputarlos.

In [4]:
missing = supervised[PM25_FEATURES + METEO_FEATURES + CALENDAR_FEATURES].isna().mean().mul(100).round(1)
display(missing.to_frame('faltante_pct').T)
complete_meteo = supervised[METEO_FEATURES].notna().all(axis=1)
print('Registros con toda la meteorología:', int(complete_meteo.sum()), f'({100 * complete_meteo.mean():.1f} %)')

,pm25_mean,pm25_lag1,pm25_lag2,pm25_lag3,pm25_lag4,pm25_lag5,pm25_lag6,pm25_roll3_mean,pm25_roll7_mean,pm25_roll7_std,...,pm25_evening_mean,t,h,pr,vv,vv_max,p,p_sum3,target_month,target_day_of_week
faltante_pct,0.0,1.6,2.3,2.6,2.9,3.0,3.0,0.7,1.5,1.5,...,0.7,7.6,8.2,17.4,10.9,11.3,9.2,11.0,0.0,0.0


Registros con toda la meteorología: 32455 (73.1 %)


Los predictores de PM2.5 tienen a lo sumo 3 % de faltantes: los rezagos solo se pierden cuando algún día previo no alcanzó 18 horas válidas. La meteorología tiene más huecos, sobre todo la presión (17,4 %), por la estación 252 sin presión desde 2019 (asignada a ITA-CJUS y ENV-HOSP). Solo el 73,1 % de los registros tiene toda la meteorología. Es algo menos que los 33.134 del EDA porque ahora se exige también la lluvia de los tres días previos y se descartan las presiones de 0 hPa. Excluir esos registros reduciría la tabla en una cuarta parte, por lo que el modelo los usará con faltantes.

## 5. Línea base: persistencia por año de prueba

La persistencia pronostica que mañana tendrá la misma media que hoy. Se evalúa en cada año de prueba del origen móvil (2022, 2023, 2024 y 2025, según el año del día `d`, como en la Tabla II del paper). Además del error se reporta R², porque es la métrica que publicaron Baena-Salazar et al., y el desempeño como aviso de Nivel de Prevención con umbral 38 µg/m³.

In [5]:
def regression_metrics(y_true, y_pred):
    error = y_true - y_pred
    return {
        'MAE': error.abs().mean(),
        'RMSE': np.sqrt((error ** 2).mean()),
        'R2': 1 - (error ** 2).sum() / ((y_true - y_true.mean()) ** 2).sum(),
    }


def warning_metrics(y_true, y_pred, threshold=PREVENTION_THRESHOLD):
    event, warning = y_true >= threshold, y_pred >= threshold
    hits = int((event & warning).sum())
    return {
        'eventos': int(event.sum()),
        'avisos': int(warning.sum()),
        'precision': hits / warning.sum() if warning.sum() else np.nan,
        'recall': hits / event.sum() if event.sum() else np.nan,
    }


rows = []
for year in TEST_YEARS:
    test = supervised[supervised['year'] == year]
    rows.append({
        'año': year,
        'registros': len(test),
        **regression_metrics(test[TARGET], test['pm25_mean']),
        **warning_metrics(test[TARGET], test['pm25_mean']),
    })
persistence = pd.DataFrame(rows).set_index('año')
display(persistence.round(3))

,registros,MAE,RMSE,R2,eventos,avisos,precision,recall
año,,,,,,,,
2022,5546,3.610,4.701,0.537,59,58,0.500,0.492
2023,5465,3.362,4.508,0.438,36,37,0.270,0.278
2024,5520,3.560,4.668,0.649,158,159,0.597,0.601
2025,5409,2.950,3.884,0.469,4,5,0.000,0.000


La persistencia comete un error medio de 2,95 a 3,61 µg/m³ según el año. Su R² va de 0,44 a 0,65, muy por debajo del 0,80 que alcanza en el periodo de validación de Baena-Salazar et al. (enero–marzo de 2016, ver `feedback-delivery-01/respuestas.md`). La razón es que el R² depende de cuánto varía la serie en el periodo evaluado: ese trimestre incluyó una contingencia ambiental con grandes cambios de nivel, mientras que un año completo contiene muchos días de concentración moderada. Por eso la comparación con ese estudio debe hacerse en su mismo periodo y no con el R² de nuestra ventana.

Como aviso de Nivel de Prevención, la persistencia acierta alrededor de la mitad de los eventos en 2022 y 2024, años con episodios de varios días, y apenas una cuarta parte en 2023, cuando la mayoría de los eventos son inicios de episodio (26 de 36). En 2025, con solo 4 eventos, no acierta ninguno: no hay suficientes eventos para medir anticipación, lo que confirma que la evaluación debe reportarse año por año y no con un único periodo de prueba.

## 6. Siguientes pasos

1. Entrenar `HistGradientBoostingRegressor` con origen móvil, sin y con meteorología, y compararlo con esta tabla.
2. Fijar el umbral de aviso con validación (sin tocar el año de prueba) y comparar con la persistencia a igual número de avisos.
3. SHAP sobre el modelo con meteorología.
4. Réplica del periodo de validación de Baena-Salazar et al. (enero–marzo de 2016).